# Setup: Generate Sample Dataset

This cell creates the required folder structure (`data/raw/` and `data/processed/`) relative to the notebook, and generates the sample CSV dataset with missing values. 
This ensures the dataset is ready for cleaning functions and saves it to `data/raw/sample_data.csv`.

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install pandas

In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    ("src/cleaning.py", "NEEDED", "YOU write this in the homework - the import fails until you do"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/paritoshdwivedi/Downloads/project bootcamp/bootcamp_paritosh_dwivedi/homework/homework06

  [OK ]  NEEDED    src/cleaning.py                     YOU write this in the homework - the import fails until you do

All needed files present.


In [3]:
import os
import pandas as pd
import numpy as np

# Define folder paths relative to this notebook
raw_dir = 'data/raw'
processed_dir = 'data/processed'

# Create folders if they don't exist
os.makedirs(raw_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

# Define the sample data
data = {
    'age': [34, 45, 29, 50, 38, np.nan, 41],
    'income': [55000, np.nan, 42000, 58000, np.nan, np.nan, 49000],
    'score': [0.82, 0.91, np.nan, 0.76, 0.88, 0.65, 0.79],
    'zipcode': ['90210', '10001', '60614', '94103', '73301', '12345', '94105'],
    'city': ['Beverly', 'New York', 'Chicago', 'SF', 'Austin', 'Unknown', 'San Francisco'],
    'extra_data': [np.nan, 42, np.nan, np.nan, np.nan, 5, np.nan]
}

# Create DataFrame
df = pd.DataFrame(data)

# Save to CSV in raw data folder
csv_path = os.path.join(raw_dir, 'sample_data.csv')
if not os.path.exists(csv_path):
    df.to_csv(csv_path, index=False)
    print(f'Sample dataset created and saved to {csv_path}')
else:
    print(f'File already exists at {csv_path}. Skipping CSV creation to avoid overwrite.')


File already exists at data/raw/sample_data.csv. Skipping CSV creation to avoid overwrite.


# Homework 06: Data preprocessing

This notebook applies reusable, copy-safe cleaning functions to the provided sample data. It keeps the raw file unchanged and writes a reproducible processed file for comparison.

The exercise also rehearses the preprocessing discipline used by the Weekly ETF Risk Monitor: separate raw and processed artifacts, distinguish measured quantities from identifiers, and state the information cost of every cleaning choice.

In [4]:
import pandas as pd

from src import cleaning

## Load the raw dataset

The raw CSV is loaded into `original` and never reassigned. Keeping that object unchanged makes the before-and-after comparison auditable.

In [5]:
raw_path = Path("data/raw/sample_data.csv")
original = pd.read_csv(raw_path)

print(f"Loaded {original.shape[0]} rows and {original.shape[1]} columns from {raw_path}")
original

Loaded 7 rows and 6 columns from data/raw/sample_data.csv


,age,income,score,zipcode,city,extra_data
0,34.0,55000.0,0.82,90210,Beverly,NaN
1,45.0,NaN,0.91,10001,New York,42.0
2,29.0,42000.0,NaN,60614,Chicago,NaN
3,50.0,58000.0,0.76,94103,SF,NaN
4,38.0,NaN,0.88,73301,Austin,NaN
5,NaN,NaN,0.65,12345,Unknown,5.0
6,41.0,49000.0,0.79,94105,San Francisco,NaN


## Apply the cleaning functions in a deliberate order

The operations are ordered to match the information value of the fields:

1. Drop columns with more than 70% missing values. `extra_data` has only two observed values out of seven, so filling it would turn a weakly observed field into a misleadingly complete feature.
2. Median-fill only `age`, `income`, and `score`. These are continuous measures. `zipcode` is an identifier even though pandas infers a numeric dtype, and `city` is categorical, so neither should be treated as a continuous variable.
3. Min-max scale the three filled measures. Filling first ensures the imputed observations are converted to the same unitless scale as observed values.

For this dataset, the median is inside the observed range, so filling does not change the original minimum or maximum. It does change the scaling result for previously missing cells: they become scaled median values instead of remaining missing. Normalizing first would require a second missing-value operation on already-scaled units and would make the policy harder to audit.

In [6]:
continuous_columns = ["age", "income", "score"]

without_sparse_columns = cleaning.drop_missing(original, threshold=0.70)
filled = cleaning.fill_missing_median(without_sparse_columns, columns=continuous_columns)
cleaned = cleaning.normalize_data(filled, columns=continuous_columns)

# Verify the raw DataFrame was not changed and the intended policy was applied.
assert "extra_data" in original.columns
assert "extra_data" not in cleaned.columns
assert original[continuous_columns].isna().sum().sum() > 0
assert cleaned[continuous_columns].isna().sum().sum() == 0
assert cleaned[continuous_columns].min().eq(0.0).all()
assert cleaned[continuous_columns].max().eq(1.0).all()

cleaned

,age,income,score,zipcode,city
0,0.238095,0.8125,0.653846,90210,Beverly
1,0.761905,0.6250,1.000000,10001,New York
2,0.000000,0.0000,0.596154,60614,Chicago
3,1.000000,1.0000,0.423077,94103,SF
4,0.428571,0.6250,0.884615,73301,Austin
5,0.500000,0.6250,0.000000,12345,Unknown
6,0.571429,0.4375,0.538462,94105,San Francisco


## Save the cleaned dataset

The processed CSV is derived entirely from the raw file and the three reusable functions. Re-running the notebook recreates the same output.

In [7]:
processed_path = Path("data/processed/sample_data_cleaned.csv")
cleaned.to_csv(processed_path, index=False)

print(f"Saved {cleaned.shape[0]} rows and {cleaned.shape[1]} columns to {processed_path}")

Saved 7 rows and 5 columns to data/processed/sample_data_cleaned.csv


## Compare the original and cleaned data

The tables below show the shape, per-column missing counts, and descriptive statistics side by side. A missing entry for `extra_data` in the cleaned column means that the field was removed, not that it has zero missing values.

In [8]:
shape_comparison = pd.DataFrame(
    {
        "original": [original.shape[0], original.shape[1]],
        "cleaned": [cleaned.shape[0], cleaned.shape[1]],
    },
    index=["rows", "columns"],
)
shape_comparison

,original,cleaned
rows,7,7
columns,6,5


In [9]:
missing_comparison = pd.concat(
    [
        original.isna().sum().rename("original"),
        cleaned.isna().sum().rename("cleaned"),
    ],
    axis=1,
).astype("Int64")
missing_comparison

,original,cleaned
age,1,0
income,3,0
score,1,0
zipcode,0,0
city,0,0
extra_data,5,<NA>


In [10]:
description_comparison = pd.concat(
    {
        "original": original.describe(include="all"),
        "cleaned": cleaned.describe(include="all"),
    },
    axis=1,
)
description_comparison

original                                                           \
              age        income     score      zipcode     city extra_data   
count    6.000000      4.000000  6.000000      7.00000        7   2.000000   
unique        NaN           NaN       NaN          NaN        7        NaN   
top           NaN           NaN       NaN          NaN  Beverly        NaN   
freq          NaN           NaN       NaN          NaN        1        NaN   
mean    39.500000  51000.000000  0.801667  62097.00000      NaN  23.500000   
std      7.556454   7071.067812  0.092826  36869.63632      NaN  26.162951   
min     29.000000  42000.000000  0.650000  10001.00000      NaN   5.000000   
25%     35.000000  47250.000000  0.767500  36479.50000      NaN  14.250000   
50%     39.500000  52000.000000  0.805000  73301.00000      NaN  23.500000   
75%     44.000000  55750.000000  0.865000  92156.50000      NaN  32.750000   
max     50.000000  58000.000000  0.910000  94105.00000      NaN  42.000000   

         cleaned                                            
             age    income     score      zipcode     city  
count   7.000000  7.000000  7.000000      7.00000        7  
unique       NaN       NaN       NaN          NaN        7  
top          NaN       NaN       NaN          NaN  Beverly  
freq         NaN       NaN       NaN          NaN        1  
mean    0.500000  0.589286  0.585165  62097.00000      NaN  
std     0.328479  0.314281  0.325952  36869.63632      NaN  
min     0.000000  0.000000  0.000000  10001.00000      NaN  
25%     0.333333  0.531250  0.480769  36479.50000      NaN  
50%     0.500000  0.625000  0.596154  73301.00000      NaN  
75%     0.666667  0.718750  0.769231  92156.50000      NaN  
max     1.000000  1.000000  1.000000  94105.00000      NaN

## Assumptions and tradeoffs

| Decision | Assumption | Tradeoff |
|---|---|---|
| Drop `extra_data` above 70% missingness | Two observations are insufficient for a stable general-purpose feature. | The two recorded values and any signal they contain are lost. The 70% cutoff is a policy choice, not a universal rule. |
| Retain all rows | In this small sample, preserving coverage matters more than removing incomplete records. | Filling introduces estimated rather than observed values. Dropping one row would lose about 14% of this sample and could bias the result if missingness is systematic. |
| Median-fill continuous measures | The observed median is a defensible center and missingness is not itself the target of this exercise. | Median filling compresses variance, weakens correlations, and can hide informative missingness. A production workflow could add missingness indicators. |
| Exclude `zipcode` and `city` from numeric cleaning | They are identifiers or categories, not quantities with meaningful arithmetic distance. | They remain unencoded and cannot be used directly by most models. Encoding would require a separate, justified policy. |
| Min-max scale `age`, `income`, and `score` | Relative position within the sample range is useful for comparison. | Min-max scaling is sensitive to outliers. New values outside the fitted range can fall below 0 or above 1. |
| Map constant numeric columns to 0.0 | A zero-range feature contains no relative variation. | The output does not preserve the original level, only the fact that all observed values were equal. |

Row dropping would be preferable when a required field is missing in only a small number of records and the missing rows are plausibly representative of the same population. Filling is preferable here because the sample is small and defensible medians exist. If missingness depended on income, age, or risk, either choice could bias the analysis and would require investigation.

For the Weekly ETF Risk Monitor, preprocessing parameters such as medians and min-max bounds must be learned only from training data, then applied unchanged to later observations. This notebook has no train-test split because it demonstrates cleaning rather than forecasting, but carrying full-sample statistics into model evaluation would create look-ahead leakage.

## Result

The cleaned dataset preserves all seven records, removes the near-empty field, fills missing continuous measurements, and scales those measurements to `[0, 1]`. The raw CSV remains the audit source, and `data/processed/sample_data_cleaned.csv` is the reproducible derived artifact.